In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

# Загружаем датасет (Хоккей против Бейсбола)
# Используем remove=(), чтобы удалить служебные заголовки email-писем и оставить только суть
cats = ['rec.sport.hockey', 'rec.sport.baseball']
train_data = fetch_20newsgroups(subset='train', categories=cats, remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', categories=cats, remove=('headers', 'footers', 'quotes'))

df_train = pd.DataFrame({'text': train_data.data, 'label': train_data.target})
df_test = pd.DataFrame({'text': test_data.data, 'label': test_data.target})

# Удаляем пустые строки, которые могли остаться после очистки заголовков
df_train = df_train[df_train['text'].str.strip().astype(bool)]
df_test = df_test[df_test['text'].str.strip().astype(bool)]

print(f"Размер обучающей выборки: {len(df_train)} текстов")
print(f"Размер тестовой выборки: {len(df_test)} текстов")

Размер обучающей выборки: 1160 текстов
Размер тестовой выборки: 773 текстов


In [6]:
def clean_text(text):
    """
    Кастомная функция предобработки текста.
    """
    text = str(text).lower() # 1. Приведение к нижнему регистру
    text = re.sub(r'\d+', '', text) # 2. Удаление цифр
    text = re.sub(r'[^\w\s]', '', text) # 3. Удаление пунктуации (оставляем только буквы и пробелы)
    
    tokens = text.split() # 4. Базовая токенизация (разбиение на слова)
    
    # 5. Удаление стоп-слов (предлоги, союзы, местоимения)
    cleaned_tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS]
    
    return " ".join(cleaned_tokens)

# Проверим работу функции на примере
sample_text = "Wow! The player #99 scored 2 amazing goals in the 3rd period..."
print("Исходный текст:", sample_text)
print("Очищенный текст:", clean_text(sample_text))

Исходный текст: Wow! The player #99 scored 2 amazing goals in the 3rd period...
Очищенный текст: wow player scored amazing goals rd period


In [7]:
# Создаем пайплайн: векторизация + классификация
pipeline = Pipeline([
    (
        'tfidf', 
        TfidfVectorizer(
            preprocessor=clean_text, 
            min_df=5,       # Слово должно встречаться минимум в 5 документах
            max_df=0.7      # Слово не должно встречаться чаще, чем в 70% документов
        )
    ),
    ('clf', MultinomialNB())
])

# ВАЖНО: Метод fit() вызывается ТОЛЬКО на обучающей выборке df_train!
pipeline.fit(df_train['text'], df_train['label'])

# На тестовой выборке вызываем только predict()
y_pred = pipeline.predict(df_test['text'])

print("Точность (Accuracy):", accuracy_score(df_test['label'], y_pred))
print("\nОтчет по классификации:\n", classification_report(
    df_test['label'], y_pred, target_names=train_data.target_names
))

Точность (Accuracy): 0.9275549805950841

Отчет по классификации:
                     precision    recall  f1-score   support

rec.sport.baseball       0.93      0.92      0.93       383
  rec.sport.hockey       0.92      0.93      0.93       390

          accuracy                           0.93       773
         macro avg       0.93      0.93      0.93       773
      weighted avg       0.93      0.93      0.93       773



In [8]:
# Находим индексы, где предсказание не совпало с реальностью
errors_mask = df_test['label'] != y_pred
df_errors = df_test[errors_mask].copy()

# ИСПРАВЛЕНИЕ: фильтруем y_pred по той же маске, чтобы размеры совпали
df_errors['predicted'] = y_pred[errors_mask]

print(f"Всего ошибок на тестовой выборке: {len(df_errors)}\n")
print("--- ПРИМЕРЫ 5 ОШИБОК ---")

# Выводим первые 5 ошибок
for idx, row in df_errors.head(5).iterrows():
    true_label = train_data.target_names[row['label']]
    pred_label = train_data.target_names[row['predicted']]
    
    print(f"[{idx}] Истина: {true_label} | Предсказание модели: {pred_label}")
    # Выводим первые 250 символов текста, чтобы понять контекст
    print(f"Текст: {row['text'][:250]}...\n")

Всего ошибок на тестовой выборке: 56

--- ПРИМЕРЫ 5 ОШИБОК ---
[6] Истина: rec.sport.baseball | Предсказание модели: rec.sport.hockey
Текст: 
I think what Bob is describing here is a game which MAD magazine
called 'Basebrawl'.  I have no idea what issue, but it sure did cover
the violence issue.

paul



...

[65] Истина: rec.sport.baseball | Предсказание модели: rec.sport.hockey
Текст: 
   And it has worked again. Pena went 3 for 3 last night against Colorado.
...

[75] Истина: rec.sport.baseball | Предсказание модели: rec.sport.hockey
Текст: 
Wait a minute; I could swear that Varsho is white....

[94] Истина: rec.sport.baseball | Предсказание модели: rec.sport.hockey
Текст: 
You sure?  This maybe a chronic syndrome.  If it persists, we may have to 
banish you to the "Cub-crazy Sanatarium" in north Chicago...
:-):-):-)...

[109] Истина: rec.sport.baseball | Предсказание модели: rec.sport.hockey
Текст: BASEBALL.....ALWAYS
how he

...



### Анализ ошибок модели (Гипотезы)

Исходя из выведенных примеров неверной классификации, можно выделить следующие причины ошибок:

1. **Слишком короткие тексты:** Если текст состоит из пары общих слов (например, просто *"Yes, I agree"*), после очистки от стоп-слов в нем не остается специфичных терминов (ключевых признаков). Модель гадает почти случайным образом.
2. **Пересечение терминологии:** Некоторые термины присутствуют в обоих видах спорта (например, *player, team, season, game, ball* часто употребляются в переносном смысле). Если в хоккейном тексте случайно нет слова *ice* или *puck*, но есть много общих слов, TF-IDF может сместить веса в сторону бейсбола.
3. **Обсуждение сразу двух тем:** В спортивных новостях часто сравнивают два вида спорта в одном абзаце (например, *"Хоккейные фанаты агрессивнее бейсбольных..."*). Модель, основанная на Bag-of-Words/TF-IDF, не понимает контекста и синтаксиса, она просто суммирует веса слов. Если бейсбольных терминов оказалось количественно больше, модель выберет бейсбол, даже если суть текста была о хоккее.
4. **Опечатки или сленг:** Специфичный фанатский сленг, который редко встречался в обучающей выборке (min_df=5), был проигнорирован векторизатором, из-за чего текст потерял свою "спортивную окраску".